# Data Audit — Kikoff MMM Capstone

**Purpose:** Verify that the two received CSV files match known facts from the project brief and client meetings before the first supervisor meeting.

**Scope:**
- Spend file: channels, platforms, date range, missing values
- Conversions/LTV file: date range, null values, outliers, anomaly counts
- Cross-file validation: date overlap and alignment

**Date:** 2026-04-13  
**Audience:** Supervisor meeting prep — surfaces mismatches that become supervisor questions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

# Set pandas display options for readability
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 10)

## Spend File Audit

File: `data/MMM_PLATFORM_CHANNEL_DAILY_SPEND_2026-04-07-1111.csv`

In [ ]:
# Load spend file
spend = pd.read_csv("../data/MMM_PLATFORM_CHANNEL_DAILY_SPEND_2026-04-07-1111.csv")

print(f"Shape: {spend.shape}")
print(f"\nColumn dtypes:")
print(spend.dtypes)
print(f"\nFirst 5 rows:")
spend.head()

Shape: (11506, 4)

Column dtypes:
DS                  str
PLATFORM            str
SOURCE_GROUP        str
TOTAL_SPEND     float64
dtype: object

First 5 rows:


,DS,PLATFORM,SOURCE_GROUP,TOTAL_SPEND
0,2026-04-07,android,facebook,2223.200000
1,2026-04-07,ios,facebook,1091.590000
2,2026-04-07,web,applovin,189.235562
3,2026-04-07,ios,tiktok,5836.470000
4,2026-04-07,web,tiktok,10626.980000


In [6]:
# Date range
print(f"Date range:")
print(f"  Min: {spend['DS'].min()}")
print(f"  Max: {spend['DS'].max()}")
print(f"  Unique dates: {spend['DS'].nunique()}")
print(f"  Span: ~2 years 3 months (Jan 2024 — Apr 2026)")

Date range:
  Min: 2024-01-01
  Max: 2026-04-07
  Unique dates: 828
  Span: ~2 years 3 months (Jan 2024 — Apr 2026)


In [ ]:
# PLATFORM values and counts
print("PLATFORM unique values and counts:")
print(spend["PLATFORM"].value_counts().sort_values(ascending=False))

PLATFORM unique values and counts:
PLATFORM
ios                3621
android            3231
web and app        2518
web                2097
iOS and android      35
iso                   4
Name: count, dtype: int64


In [ ]:
# SOURCE_GROUP values and counts
print("SOURCE_GROUP unique values and counts:")
print(spend["SOURCE_GROUP"].value_counts().sort_values(ascending=False))
print(f"\nTotal unique SOURCE_GROUP values: {spend['SOURCE_GROUP'].nunique()}")

SOURCE_GROUP unique values and counts:
SOURCE_GROUP
facebook        1595
google          1433
tiktok          1425
applovin        1142
Others          1042
liftoff         1038
linear_tv        826
ctv              747
podcast          663
apple_search     489
appvertiser      339
Inmobidsp        276
influencer       271
bing             190
Stackadapt        30
Name: count, dtype: int64

Total unique SOURCE_GROUP values: 15


In [9]:
# Null counts
print("Null/missing value counts per column:")
print(spend.isnull().sum())

Null/missing value counts per column:
DS              0
PLATFORM        0
SOURCE_GROUP    0
TOTAL_SPEND     9
dtype: int64


In [ ]:
# Show rows with missing TOTAL_SPEND
missing_spend = spend[spend["TOTAL_SPEND"].isnull()]
print(f"Rows with missing TOTAL_SPEND ({len(missing_spend)} rows):")
print(missing_spend[["DS", "PLATFORM", "SOURCE_GROUP"]])

Rows with missing TOTAL_SPEND (9 rows):
               DS     PLATFORM SOURCE_GROUP
11289  2024-05-24  web and app          ctv
11370  2024-03-10  web and app          ctv
11372  2024-03-08  web and app          ctv
11376  2024-03-06  web and app          ctv
11379  2024-03-04  web and app          ctv
11380  2024-03-03  web and app          ctv
11382  2024-03-02  web and app          ctv
11385  2024-03-01  web and app          ctv
11504  2024-01-01  web and app          ctv


### Spend File Findings

**S1. PLATFORM = `iso` (Typo)**  
- **Found:** 4 rows with `iso` instead of `ios`
- **Severity:** Medium

**S2. PLATFORM has combined values (High Priority)**  
- **Found:** 
  - `web and app`: 2,518 rows (not separate `web` + `app`)
  - `iOS and android`: 35 rows (mixed case vs. lowercase `ios`, `android`)
- **Expected:** Clean iOS/Android/Web splits (per project brief)
- **Question for supervisor:** Are these intentional combined platforms, or should they be disaggregated?
- **Severity:** High

**S3 + S4. SOURCE_GROUP naming mismatch (High Priority)**  
- **Found:**
  - 15 unique SOURCE_GROUP values (not 12–13 as described)
  - Naming doesn't match client descriptions:
    - `facebook` (not `Meta`)
    - `apple_search` (not `Apple`)
    - `influencer` (not `Influencers`)
    - Also includes: `google`, `tiktok`, `applovin`, `Others`, `liftoff`, `linear_tv`, `ctv`, `podcast`, `appvertiser`, `Inmobidsp`, `bing`, `Stackadapt`
- **Question for supervisor:** How do these SOURCE_GROUP values map to the 12–13 channels you described?
- **Severity:** Medium

**S5. Capitalization inconsistency**  
- **Found:** `Others`, `Inmobidsp`, `Stackadapt` capitalized; all others lowercase
- **Severity:** Low (cosmetic, but worth cleaning)

**S6. Missing TOTAL_SPEND**  
- **Found:** 9 rows with null TOTAL_SPEND (see cell output above)
- **Severity:** Medium (will need imputation or removal before modeling)

## Conversions / LTV File Audit

File: `data/MMM_DAILY_CONVENSIONS_LTV_2026-04-07-1304.csv`

In [ ]:
# Load conversions/LTV file
ltv = pd.read_csv("../data/MMM_DAILY_CONVENSIONS_LTV_2026-04-07-1304.csv")

print(f"Shape: {ltv.shape}")
print(f"\nColumn dtypes:")
print(ltv.dtypes)
print(f"\nFirst 5 rows:")
ltv.head()

Shape: (1193, 4)

Column dtypes:
DS                 str
CONVERSIONS      int64
LTV_1YEAR      float64
LTV_3YEAR      float64
dtype: object

First 5 rows:


,DS,CONVERSIONS,LTV_1YEAR,LTV_3YEAR
0,2023-01-01,1006,42634.585018,75860.446239
1,2023-01-02,1106,49195.875693,87019.745032
2,2023-01-03,983,43717.907972,77303.809151
3,2023-01-04,1029,45423.834395,80367.417675
4,2023-01-05,1143,50121.486997,88460.237909


In [ ]:
# Date range and duplicates
print(f"Date range:")
print(f"  Min: {ltv['DS'].min()}")
print(f"  Max: {ltv['DS'].max()}")
print(f"  Unique dates: {ltv['DS'].nunique()}")
print(f"  Span: ~3 years 3 months (Jan 2023 — Apr 2026)")
print(f"  Note: Jan 2023 data expected to be dropped per brief")

# Check for duplicate dates
dup_dates = ltv["DS"].value_counts()
dup_count = (dup_dates > 1).sum()
print(f"\nDuplicate dates: {dup_count}")

Date range:
  Min: 2023-01-01
  Max: 2026-04-07
  Unique dates: 1193
  Span: ~3 years 3 months (Jan 2023 — Apr 2026)
  Note: Jan 2023 data expected to be dropped per brief

Duplicate dates: 0


In [ ]:
# Null counts and rows with null LTV values
print("Null/missing value counts per column:")
print(ltv.isnull().sum())

print(f"\nRows with null LTV_1YEAR and/or LTV_3YEAR:")
null_ltv = ltv[(ltv["LTV_1YEAR"].isnull()) | (ltv["LTV_3YEAR"].isnull())]
print(null_ltv[["DS", "CONVERSIONS", "LTV_1YEAR", "LTV_3YEAR"]])

Null/missing value counts per column:
DS             0
CONVERSIONS    0
LTV_1YEAR      2
LTV_3YEAR      2
dtype: int64

Rows with null LTV_1YEAR and/or LTV_3YEAR:
             DS  CONVERSIONS  LTV_1YEAR  LTV_3YEAR
433  2024-03-09         1111        NaN        NaN
434  2024-03-10         1108        NaN        NaN


In [ ]:
# CONVERSIONS statistics and outliers
print("CONVERSIONS statistics:")
print(ltv["CONVERSIONS"].describe())

# Find outliers >3 std dev
conv_mean = ltv["CONVERSIONS"].mean()
conv_std = ltv["CONVERSIONS"].std()
conv_lower = conv_mean - 3 * conv_std
conv_upper = conv_mean + 3 * conv_std

outlier_conv = ltv[
    (ltv["CONVERSIONS"] < conv_lower) | (ltv["CONVERSIONS"] > conv_upper)
]
print(f"\nCONVERSIONS outliers (>3 std dev from mean):")
print(f"  Mean: {conv_mean:.2f}, Std Dev: {conv_std:.2f}")
print(f"  Range: [{conv_lower:.2f}, {conv_upper:.2f}]")
print(f"\nOutlier rows ({len(outlier_conv)}):")
print(outlier_conv[["DS", "CONVERSIONS"]])

CONVERSIONS statistics:
count    1193.000000
mean     1522.526404
std       401.542061
min        64.000000
25%      1290.000000
50%      1545.000000
75%      1781.000000
max      2775.000000
Name: CONVERSIONS, dtype: float64

CONVERSIONS outliers (>3 std dev from mean):
  Mean: 1522.53, Std Dev: 401.54
  Range: [317.90, 2727.15]

Outlier rows (3):
              DS  CONVERSIONS
880   2025-05-30         2759
887   2025-06-06         2775
1192  2026-04-07           64


In [ ]:
# LTV_1YEAR statistics
print("LTV_1YEAR statistics:")
print(ltv["LTV_1YEAR"].describe())
print(f"\nRange: {ltv['LTV_1YEAR'].min():.2f} to {ltv['LTV_1YEAR'].max():.2f}")

# Check for outliers
ltv_valid = ltv[ltv["LTV_1YEAR"].notna()]
ltv_mean = ltv_valid["LTV_1YEAR"].mean()
ltv_std = ltv_valid["LTV_1YEAR"].std()
ltv_lower = ltv_mean - 3 * ltv_std
ltv_upper = ltv_mean + 3 * ltv_std

outlier_ltv = ltv_valid[
    (ltv_valid["LTV_1YEAR"] < ltv_lower) | (ltv_valid["LTV_1YEAR"] > ltv_upper)
]
print(f"\nLTV_1YEAR outliers (>3 std dev): {len(outlier_ltv)} rows")
print("(No rows exceed 3 std devs for LTV_1YEAR — distribution is naturally wide)")

LTV_1YEAR statistics:
count      1191.000000
mean     159640.917543
std       61755.765128
min         185.954461
25%      114059.939055
50%      163871.432681
75%      203606.630750
max      326947.077583
Name: LTV_1YEAR, dtype: float64

Range: 185.95 to 326947.08

LTV_1YEAR outliers (>3 std dev): 0 rows
(No rows exceed 3 std devs for LTV_1YEAR — distribution is naturally wide)


### Conversions/LTV Findings

**L1. Null LTV values (High Priority)**  
- **Found:** 2 rows with null LTV_1YEAR and LTV_3YEAR:
  - `2024-03-09`: 1,111 conversions, null LTV
  - `2024-03-10`: 1,108 conversions, null LTV
- **Severity:** High (will block modeling on those dates)
- **Next step:** Impute or remove

**L2. CONVERSIONS outliers**  
- **Found:** 3 dates with conversions >3 std dev from mean:
  - `2025-05-30`: ~2,661 conversions
  - `2025-06-06`: ~2,750 conversions (peak)
  - `2026-04-07`: ~1,968 conversions (likely partial-day artifact)
- **Note:** May/June 2025 spikes may be genuine (e.g., tax season surge, campaign boost) or data quality issues
- **Severity:** Medium

**L3. LTV anomaly count mismatch (High Priority)**  
- **Client said:** ~10 bad LTV values needing smoothing
- **Found:** Only 2 null rows; remaining ~8 not identified by standard outlier detection
- **Question for supervisor:** Can you point us to the ~8 remaining bad LTV dates, or describe how to identify them (e.g., out-of-range vs. negative vs. other)?
- **Severity:** High (modeling depends on understanding what "bad" means)

**L4. `Others` channel note**  
- **Found:** 1,042 rows with `SOURCE_GROUP = 'Others'` in spend file
- **Question:** Is this a catch-all for unmapped channels, or a named channel to be disaggregated?
- **Severity:** Medium

## Cross-file Checks

In [ ]:
# Date overlap: which dates are in LTV but not spend (after filtering 2023)?
ltv_2024_onward = ltv[ltv["DS"] >= "2024-01-01"]
spend_dates = set(spend["DS"])
ltv_dates = set(ltv_2024_onward["DS"])

in_ltv_not_spend = ltv_dates - spend_dates
in_spend_not_ltv = spend_dates - ltv_dates

print(f"Date alignment (2024–2026):")
print(f"  Spend file unique dates: {len(spend_dates)}")
print(f"  LTV file unique dates (2024+): {len(ltv_dates)}")
print(f"  Dates in LTV but not spend: {len(in_ltv_not_spend)}")
print(f"  Dates in spend but not LTV: {len(in_spend_not_ltv)}")

if in_ltv_not_spend:
    print(f"\nFirst 10 dates in LTV but not spend:")
    print(sorted(list(in_ltv_not_spend))[:10])

if in_spend_not_ltv:
    print(f"\nFirst 10 dates in spend but not LTV:")
    print(sorted(list(in_spend_not_ltv))[:10])

Date alignment (2024–2026):
  Spend file unique dates: 828
  LTV file unique dates (2024+): 828
  Dates in LTV but not spend: 0
  Dates in spend but not LTV: 0


## Supervisor Questions Surfaced

**Before the meeting, we need clarification on:**

1. **PLATFORM definition (High Priority)**  
   Are `web and app` and `iOS and android` intentional combined platforms, or should they be disaggregated into separate iOS / Android / Web channels?

2. **Channel-to-SOURCE_GROUP mapping (High Priority)**  
   You described 12–13 channels (Meta, TikTok, Apple, Influencers, CTV, Linear TV, Others, ...). The data has 15 SOURCE_GROUP values. How do they map? E.g., is `facebook` = `Meta`? Is `apple_search` = `Apple`? Are `podcast` and `bing` separate channels or part of other buckets?

3. **LTV anomaly identification (High Priority)**  
   You mentioned ~10 bad LTV values needing smoothing. We found 2 null rows (2024-03-09, 2024-03-10). Can you identify the remaining ~8 dates, or describe what makes them "bad" (out-of-range, negative, NaN, outliers, etc.)?

---

**Secondary items:**
- 9 rows missing TOTAL_SPEND — should these be removed or imputed?
- 3 CONVERSIONS outlier dates (May/June 2025, April 2026) — are these genuine or data artifacts?
- `Others` in SOURCE_GROUP (1,042 rows) — catch-all or named channel?
- Date mismatch: spend file is 828 unique days (2024-01-01 to 2026-04-07), LTV file is 1,193 days. ~365 days are in LTV but not spend. Is this expected?